# Phase 7 — Confidence calibration (temperature scaling)

**Nothing is trained here.** This notebook reads the `predictions.csv` files that training already saved
(the model's raw outputs — **logits** — for validation and test images). No GPU needed; runs in under a minute.

**The problem.** The model turns a logit into a probability with `p = sigmoid(logit)`. Neural networks are usually
**overconfident**: they say "99% sure" on images where they are right only ~90% of the time.

**The fix: temperature scaling.** Learn one number `T` and use `p = sigmoid(logit / T)` instead.
- `T > 1` pulls probabilities towards 50% → *less* confident.
- `T < 1` pushes them towards 0% / 100% → *more* confident.
- `T = 1` changes nothing.

Dividing by `T` never changes the **sign** of a logit, so at the 0.5 cut-off **no label changes** — only the confidence number does.

**The rule we follow.** `T` is learned on the **validation** images of each round and then *checked* on that round's
**test** images. Test images are never used to choose `T`.

## 1. Connect to Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
DRIVE_ROOT = Path("/content/drive/MyDrive/ANA_PosNeg")

## 2. Settings and loading the saved predictions

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize, minimize_scalar

MODEL = "efficientnet_b0"          # the model chosen in Phase 6
N_BINS = 10                        # confidence bins for ECE and the reliability diagram

MODEL_DIR = DRIVE_ROOT / "outputs" / MODEL
SAVE_DIR = DRIVE_ROOT / "outputs" / "calibration"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

preds = {}
for rdir in sorted(MODEL_DIR.glob("round_*")):
    if (rdir / "predictions.csv").exists():
        preds[int(rdir.name.split("_")[1])] = pd.read_csv(rdir / "predictions.csv")
print(f"{MODEL}: {len(preds)} of 5 rounds found")
for k, df in preds.items():
    print(f"  round {k}: {(df.split == 'val').sum()} validation, {(df.split == 'test').sum()} test images")

## 3. The measuring tools

- **Confidence** of one prediction = `p` if the label is Positive, `1 − p` if Negative (always between 50% and 100%).
- **NLL (negative log-likelihood, "log loss")** — punishes confident mistakes very hard. Lower is better. This is what we minimise to find `T`.
- **ECE (Expected Calibration Error)** — sort predictions into 10 confidence bins (50–55%, 55–60%, …, 95–100%).
  In each bin compare *average confidence* with *actual accuracy*. ECE is the average gap, weighted by how many
  images are in each bin. **0 = perfectly honest confidence.** Lower is better.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def nll(logits, labels):
    # binary cross-entropy written in a numerically safe way
    return float(np.mean(np.logaddexp(0, logits) - labels * logits))


def confidence_and_correct(logits, labels):
    p = sigmoid(logits)
    pred = (p >= 0.5).astype(int)
    return np.where(pred == 1, p, 1 - p), (pred == labels).astype(float)


def bin_table(logits, labels, n_bins=N_BINS):
    # one row per confidence bin: how many images, their average confidence, their actual accuracy
    conf, correct = confidence_and_correct(logits, labels)
    edges = np.linspace(0.5, 1.0, n_bins + 1)
    idx = np.clip(np.digitize(conf, edges[1:-1]), 0, n_bins - 1)
    rows = []
    for b in range(n_bins):
        m = idx == b
        rows.append(dict(bin=f"{edges[b]:.2f}-{edges[b + 1]:.2f}", n=int(m.sum()),
                         confidence=conf[m].mean() if m.any() else np.nan,
                         accuracy=correct[m].mean() if m.any() else np.nan))
    return pd.DataFrame(rows)


def ece(logits, labels, n_bins=N_BINS):
    t = bin_table(logits, labels, n_bins).dropna()
    return float((t["n"] / t["n"].sum() * (t["confidence"] - t["accuracy"]).abs()).sum())


def fit_temperature(logits, labels):
    # the single T that minimises NLL; searched on log(T) so that T stays positive
    res = minimize_scalar(lambda a: nll(logits / np.exp(a), labels), bounds=(-3, 3), method="bounded")
    return float(np.exp(res.x))

## 4. Learn `T` on validation, check it on test — per round

Each round has its own model, so each round gets its own `T`.

In [ ]:
rows, pooled = [], []
for k, df in preds.items():
    val, test = df[df.split == "val"], df[df.split == "test"]
    vz, vy = val["logit"].to_numpy(), val["label"].to_numpy()
    tz, ty = test["logit"].to_numpy(), test["label"].to_numpy()

    T = fit_temperature(vz, vy)                                   # <- validation only
    assert ((tz >= 0) == (tz / T >= 0)).all()                     # no label changes at the 0.5 cut-off
    (MODEL_DIR / f"round_{k}" / "temperature.json").write_text(json.dumps({"temperature": T}, indent=2))

    rows.append(dict(round=k, T=T,
                     val_ece_before=ece(vz, vy), val_ece_after=ece(vz / T, vy),
                     test_ece_before=ece(tz, ty), test_ece_after=ece(tz / T, ty),
                     test_nll_before=nll(tz, ty), test_nll_after=nll(tz / T, ty)))
    pooled.append(test.assign(round=k, logit_cal=tz / T))

res = pd.DataFrame(rows).set_index("round")
pooled = pd.concat(pooled)                # every image appears exactly once as a test image
display(res.round(3))
print("mean ± std:")
for c in res.columns:
    print(f"  {c:16s} {res[c].mean():.3f} ± {res[c].std():.3f}")
res.to_csv(SAVE_DIR / "calibration_per_round.csv")

## 5. All test images together: before vs after

Every image is a test image in exactly one round, so we can pool all 2,073 test predictions
(each scaled with its own round's `T`).

How to read the summary: if *average confidence* is higher than *accuracy*, the model is **overconfident**.

In [ ]:
y = pooled["label"].to_numpy()
summary = []
for name, col in (("before (raw)", "logit"), ("after (temperature)", "logit_cal")):
    z = pooled[col].to_numpy()
    conf, correct = confidence_and_correct(z, y)
    summary.append({"": name, "accuracy": correct.mean(), "average confidence": conf.mean(),
                    "gap": conf.mean() - correct.mean(), "ECE": ece(z, y), "NLL": nll(z, y),
                    "images at >= 99% confidence": int((conf >= 0.99).sum()),
                    "…of which wrong": int(((conf >= 0.99) & (correct == 0)).sum())})
summary = pd.DataFrame(summary).set_index("")
display(summary.round(4))
summary.to_csv(SAVE_DIR / "calibration_summary.csv")

print("\nPer confidence bin — BEFORE");  display(bin_table(pooled["logit"].to_numpy(), y).round(3))
print("Per confidence bin — AFTER");     display(bin_table(pooled["logit_cal"].to_numpy(), y).round(3))

## 6. Reliability diagram

- **Left:** each dot is one confidence bin. The dashed diagonal is perfect honesty (says 80% → right 80% of the time).
  Dots **below** the line = overconfident; **above** = underconfident. Bins with fewer than 10 images are hidden (too noisy).
- **Right:** how many images fall in each bin (log scale).

In [ ]:
BLUE, ORANGE = "#2a78d6", "#eb6834"
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2))
a1.plot([0.5, 1], [0.5, 1], ls="--", color="grey", lw=1, label="perfect calibration")
centres = np.linspace(0.5, 1.0, N_BINS + 1)[:-1] + 0.25 / N_BINS
for (name, col, colour), off in zip((("before", "logit", BLUE), ("after temperature scaling", "logit_cal", ORANGE)),
                                    (-0.008, 0.008)):
    z = pooled[col].to_numpy()
    t = bin_table(z, y)
    ok = t["n"] >= 10
    a1.plot(t.loc[ok, "confidence"], t.loc[ok, "accuracy"], "o-", color=colour, lw=2, ms=6,
            label=f"{name} (ECE {ece(z, y):.3f})")
    a2.bar(centres + off, t["n"].clip(lower=0.8), width=0.016, color=colour, label=name)
a1.set_xlabel("confidence the model reports"); a1.set_ylabel("how often it is actually right")
a1.set_title("Reliability diagram — pooled test images", loc="left")
a1.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.16))   # below the plot: never covers a dot
a2.set_yscale("log"); a2.set_xlabel("confidence"); a2.set_ylabel("number of images (log scale)")
a2.set_title("Where the predictions sit", loc="left"); a2.legend(frameon=False)
for a in (a1, a2):
    a.spines[["top", "right"]].set_visible(False); a.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(SAVE_DIR / "reliability_diagram.png", dpi=200, bbox_inches="tight")
plt.show()

## 7. What the user of the system will see

A few test images with the label and the confidence before and after calibration.
The label never changes; only the confidence does.

In [ ]:
def describe(z):
    p = sigmoid(z)
    return f"{'ANA-Positive' if p >= 0.5 else 'ANA-Negative'} - confidence {max(p, 1 - p):.1%}"

show = pooled.assign(abs_logit=pooled["logit"].abs()).sort_values("abs_logit")
show = pd.concat([show.head(5), show.iloc[len(show) // 2: len(show) // 2 + 3], show.tail(2)])   # unsure, typical, very sure
display(pd.DataFrame({
    "image": show["image_path"].values,
    "truth": np.where(show["label"] == 1, "Positive", "Negative"),
    "before": [describe(z) for z in show["logit"]],
    "after": [describe(z) for z in show["logit_cal"]],
}))

## 8. For information only — is the model also *shifted*?

Temperature scaling can only stretch or squeeze confidence. It cannot fix a model that systematically leans
towards one class. Our training used `pos_weight ≈ 0.39` (each positive image counted less), which can push every
logit slightly towards "Negative" — and we did see sensitivity below specificity in most rounds.

Here we fit **two** numbers on validation — a temperature `T` **and** a shift `b`: `p = sigmoid(logit / T + b)`.
A clearly positive `b` means the model under-calls positives.

**This is not adopted.** A shift moves images across the 0.5 line, so it changes labels — it is the same thing as
choosing a different cut-off, which is a separate decision. It is shown so we can discuss it with numbers.

In [ ]:
def metrics_at_half(z, y):
    pred = (z >= 0).astype(int)
    sens = ((pred == 1) & (y == 1)).sum() / (y == 1).sum()
    spec = ((pred == 0) & (y == 0)).sum() / (y == 0).sum()
    return sens, spec


rows2 = []
for k, df in preds.items():
    val, test = df[df.split == "val"], df[df.split == "test"]
    vz, vy = val["logit"].to_numpy(), val["label"].to_numpy()
    tz, ty = test["logit"].to_numpy(), test["label"].to_numpy()
    T = res.loc[k, "T"]
    opt = minimize(lambda a: nll(vz / np.exp(a[0]) + a[1], vy), x0=[np.log(T), 0.0], method="Nelder-Mead")
    T2, b = float(np.exp(opt.x[0])), float(opt.x[1])
    s1, c1 = metrics_at_half(tz / T, ty)
    s2, c2 = metrics_at_half(tz / T2 + b, ty)
    rows2.append(dict(round=k, T=T2, shift_b=b, labels_changed=int(((tz / T2 + b >= 0) != (tz >= 0)).sum()),
                      test_ece_T_only=ece(tz / T, ty), test_ece_T_and_shift=ece(tz / T2 + b, ty),
                      sens_T_only=s1, sens_T_and_shift=s2, spec_T_only=c1, spec_T_and_shift=c2))
res2 = pd.DataFrame(rows2).set_index("round")
display(res2.round(3))
print("mean:")
print(res2.mean().round(3).to_string())
res2.to_csv(SAVE_DIR / "shift_check_per_round.csv")
print("\nAll files saved to", SAVE_DIR)